# Using Machine Learning techniques to predict oral cancer

## Imports

In [287]:
import kagglehub
import os
import pandas as pd
import numpy as np

## Load in data, basic data information

In [288]:
path = kagglehub.dataset_download("menegidio/oral-cancer-prediction-dataset")

file_path = os.path.join(path, "oral_cancer_prediction_dataset.csv")
df = pd.read_csv(file_path)

df.head()

,ID,Country,Age,Gender,Tobacco Use,Alcohol Consumption,HPV Infection,Betel Quid Use,Chronic Sun Exposure,Poor Oral Hygiene,...,Difficulty Swallowing,White or Red Patches in Mouth,Tumor Size (cm),Cancer Stage,Treatment Type,"Survival Rate (5-Year, %)",Cost of Treatment (USD),Economic Burden (Lost Workdays per Year),Early Diagnosis,Oral Cancer (Diagnosis)
0,1,Italy,36,Female,Yes,Yes,Yes,No,No,Yes,...,No,No,0.000000,0,No Treatment,100.000000,0.00,0,No,No
1,2,Japan,64,Male,Yes,Yes,Yes,No,Yes,Yes,...,No,No,1.782186,1,No Treatment,83.340103,77772.50,177,No,Yes
2,3,UK,37,Female,No,Yes,No,No,Yes,Yes,...,No,Yes,3.523895,2,Surgery,63.222871,101164.50,130,Yes,Yes
3,4,Sri Lanka,55,Male,Yes,Yes,No,Yes,No,Yes,...,No,No,0.000000,0,No Treatment,100.000000,0.00,0,Yes,No
4,5,South Africa,68,Male,No,No,No,No,No,Yes,...,No,No,2.834789,3,No Treatment,44.293199,45354.75,52,No,Yes


In [289]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84922 entries, 0 to 84921
Data columns (total 25 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   ID                                        84922 non-null  int64  
 1   Country                                   84922 non-null  object 
 2   Age                                       84922 non-null  int64  
 3   Gender                                    84922 non-null  object 
 4   Tobacco Use                               84922 non-null  object 
 5   Alcohol Consumption                       84922 non-null  object 
 6   HPV Infection                             84922 non-null  object 
 7   Betel Quid Use                            84922 non-null  object 
 8   Chronic Sun Exposure                      84922 non-null  object 
 9   Poor Oral Hygiene                         84922 non-null  object 
 10  Diet (Fruits & Vegetables Intake) 

## Data cleaning

In [290]:
df.columns

Index(['ID', 'Country', 'Age', 'Gender', 'Tobacco Use', 'Alcohol Consumption',
       'HPV Infection', 'Betel Quid Use', 'Chronic Sun Exposure',
       'Poor Oral Hygiene', 'Diet (Fruits & Vegetables Intake)',
       'Family History of Cancer', 'Compromised Immune System', 'Oral Lesions',
       'Unexplained Bleeding', 'Difficulty Swallowing',
       'White or Red Patches in Mouth', 'Tumor Size (cm)', 'Cancer Stage',
       'Treatment Type', 'Survival Rate (5-Year, %)',
       'Cost of Treatment (USD)', 'Economic Burden (Lost Workdays per Year)',
       'Early Diagnosis', 'Oral Cancer (Diagnosis)'],
      dtype='object')

### Dropping columns

In this stage we get rid of features that are synonymous with cancer (eg. Tumor Size, Cancer Stage).

In [291]:
df = df.drop(df.columns[17:-2], axis=1)
df.head()

,ID,Country,Age,Gender,Tobacco Use,Alcohol Consumption,HPV Infection,Betel Quid Use,Chronic Sun Exposure,Poor Oral Hygiene,Diet (Fruits & Vegetables Intake),Family History of Cancer,Compromised Immune System,Oral Lesions,Unexplained Bleeding,Difficulty Swallowing,White or Red Patches in Mouth,Early Diagnosis,Oral Cancer (Diagnosis)
0,1,Italy,36,Female,Yes,Yes,Yes,No,No,Yes,Low,No,No,No,No,No,No,No,No
1,2,Japan,64,Male,Yes,Yes,Yes,No,Yes,Yes,High,No,No,No,Yes,No,No,No,Yes
2,3,UK,37,Female,No,Yes,No,No,Yes,Yes,Moderate,No,No,No,No,No,Yes,Yes,Yes
3,4,Sri Lanka,55,Male,Yes,Yes,No,Yes,No,Yes,Moderate,No,No,Yes,No,No,No,Yes,No
4,5,South Africa,68,Male,No,No,No,No,No,Yes,High,No,No,No,No,No,No,No,Yes


### Checking for NaN values

We check if there are any values missing from the dataset.
- `df.isna()` – returns a DataFrame of True/False values (True if the value in the original DataFrame is NaN, False otherwise)  
- `df.isna().any()` – for each column, checks if it contains at least one True value  
- `df.isna().any().any()` – returns True if there is **at least one NaN** in the original DataFrame, False otherwise

In [292]:
df.isna().any().any()

False

There are no NaN values as is, so we may proceed.

### Checking data for unique values

Before modifying the data to be fit for training we should check how many unique values each feature can have, so that we can determine how to represent it (eg. use 0 and 1 for yes-no columns or one-hot encoding for those with more posiible values).

In [293]:
for column in df.columns:
    print(f"{column} values: {df[column].unique()}")

ID values: [    1     2     3 ... 84920 84921 84922]
Country values: ['Italy' 'Japan' 'UK' 'Sri Lanka' 'South Africa' 'Taiwan' 'USA' 'Germany'
 'France' 'Australia' 'Brazil' 'Pakistan' 'Kenya' 'Russia' 'Nigeria'
 'Egypt' 'India']
Age values: [ 36  64  37  55  68  70  41  53  62  50  65  34  56  59  43  63  44  71
  51  47  58  57  54  67  31  66  48  61  46  49  60  74  42  73  69  35
  52  39  40  45  28  38  33  75  78  72  76  29  80  32  26  77  30  79
  82  89  23  22  81  18  24  83  25  86  21  87  19  27  17  85  84  20
  88  15  93  92  94  90  96  16  91 101  98]
Gender values: ['Female' 'Male']
Tobacco Use values: ['Yes' 'No']
Alcohol Consumption values: ['Yes' 'No']
HPV Infection values: ['Yes' 'No']
Betel Quid Use values: ['No' 'Yes']
Chronic Sun Exposure values: ['No' 'Yes']
Poor Oral Hygiene values: ['Yes' 'No']
Diet (Fruits & Vegetables Intake) values: ['Low' 'High' 'Moderate']
Family History of Cancer values: ['No' 'Yes']
Compromised Immune System values: ['No' 'Yes']


### Modifying the data

In [294]:
# Binary to 1/0
df["Gender"] = df["Gender"].map({"Male": 1, "Female": 0})

for column in df.columns:
    if np.isin("Yes", df[column].unique()):
        df[column] = df[column].map({"Yes": 1, "No": 0})

# One-Hot Encoding
df = pd.get_dummies(df, columns=["Country", "Diet (Fruits & Vegetables Intake)"])

# Convert all bool columns from OHE to int
bool_cols = df.select_dtypes(include="bool").columns
df[bool_cols] = df[bool_cols].astype(int)

df.head()

,ID,Age,Gender,Tobacco Use,Alcohol Consumption,HPV Infection,Betel Quid Use,Chronic Sun Exposure,Poor Oral Hygiene,Family History of Cancer,...,Country_Pakistan,Country_Russia,Country_South Africa,Country_Sri Lanka,Country_Taiwan,Country_UK,Country_USA,Diet (Fruits & Vegetables Intake)_High,Diet (Fruits & Vegetables Intake)_Low,Diet (Fruits & Vegetables Intake)_Moderate
0,1,36,0,1,1,1,0,0,1,0,...,0,0,0,0,0,0,0,0,1,0
1,2,64,1,1,1,1,0,1,1,0,...,0,0,0,0,0,0,0,1,0,0
2,3,37,0,0,1,0,0,1,1,0,...,0,0,0,0,0,1,0,0,0,1
3,4,55,1,1,1,0,1,0,1,0,...,0,0,0,1,0,0,0,0,0,1
4,5,68,1,0,0,0,0,0,1,0,...,0,0,1,0,0,0,0,1,0,0


In [295]:
# Move label to the rightmost side of the dataframe
cols = [c for c in df.columns if c != "Oral Cancer (Diagnosis)"] + ["Oral Cancer (Diagnosis)"]
df = df[cols]

### Renaming the columns

In this step we standardize the naming convention of features in the dataframe.

In [296]:
def get_renamed_column(name):
    # Remove parentheses and their contents
    name = name[:name.index("(")] + name[name.index(")") + 1:] if "(" in name else name
    name = name.strip()
    name = name.lower()
    name = name.replace(" ", "_")
    name = name.replace("__", "_")

    return name


def get_rename_dict(column_names):
    rename_dict = {}
    for name in column_names:
        rename_dict[name] = get_renamed_column(name)

    return rename_dict

In [297]:
df = df.rename(columns=get_rename_dict(df.columns))
df.head()

,id,age,gender,tobacco_use,alcohol_consumption,hpv_infection,betel_quid_use,chronic_sun_exposure,poor_oral_hygiene,family_history_of_cancer,...,country_russia,country_south_africa,country_sri_lanka,country_taiwan,country_uk,country_usa,diet_high,diet_low,diet_moderate,oral_cancer
0,1,36,0,1,1,1,0,0,1,0,...,0,0,0,0,0,0,0,1,0,0
1,2,64,1,1,1,1,0,1,1,0,...,0,0,0,0,0,0,1,0,0,1
2,3,37,0,0,1,0,0,1,1,0,...,0,0,0,0,1,0,0,0,1,1
3,4,55,1,1,1,0,1,0,1,0,...,0,0,1,0,0,0,0,0,1,0
4,5,68,1,0,0,0,0,0,1,0,...,0,1,0,0,0,0,1,0,0,1


### Check for duplicates

In [298]:
any(df.duplicated())

False

There are only unique records in the dataset already, we may proceed.

### Check for class imbalance

There should be a similar number of records in each class.

In [305]:
df[df.columns[-1]].sum() / df[df.columns[-1]].size

0.4986811426956501

Positive diagnoses make up close to 50% of all, so they are evenly balanced.